# 02 — From raw trace to a human-auditable Decision Chain

**Question:** can a reviewer follow the consequential choices without losing the
ability to inspect what actually happened?

This notebook opens one completed chart review at four levels:

1. Codex protocol events — very detailed harness traffic.
2. Canonical Langtrace/Layer-1 events — observable tool calls and server facts.
3. Deterministic ReAct cycles — state before, action, observation, state after.
4. Decision Episodes — one material choice that one human can judge with one verdict.

The Decision Chain is an index over the raw trace, not a replacement for it. Every
episode must retain a path back to its cycle and event evidence. We never display or
claim private chain-of-thought.


## 1. Select a run

The checked demonstration uses the SYNX03 policy-guided Luna run because it contains
the full pattern a reviewer needs to see: inventory, keyword search, candidate-note
selection, three independent evidence judgments, conflict resolution, and a stopping
decision. Override the paths with `ACR_AUDIT_LEDGER` and `ACR_AUDIT_RUN_ID`.


In [1]:
from collections import Counter, defaultdict
from pathlib import Path
import json
import os

from IPython.display import Markdown, display
from acr.mvp.human_review import human_review_view
from acr.mvp.ledger import SemanticaLedger

START_DIR = Path.cwd().resolve()
ROOT = START_DIR if (START_DIR / "pyproject.toml").is_file() else START_DIR.parent
assert (ROOT / "pyproject.toml").is_file(), "Start Jupyter from the repo or notebooks/"

checked_ledger = ROOT / "runs/notebook-live-20260827/ledger.json"
generated_ledger = ROOT / "runs/postdoc-study/ledger.json"
seed_ledger = ROOT / "runs/policy-experiment-20260827/experiment-ledger.json"
default_ledger = next(
    (path for path in (checked_ledger, generated_ledger, seed_ledger) if path.is_file()),
    generated_ledger,
)
LEDGER_PATH = Path(os.environ.get("ACR_AUDIT_LEDGER", default_ledger))
assert LEDGER_PATH.is_file(), "Run Notebook 1 first or set ACR_AUDIT_LEDGER"
ledger = SemanticaLedger(LEDGER_PATH)

def one_line(value, limit=100):
    text = " ".join(str("" if value is None else value).split())
    return text if len(text) <= limit else text[: limit - 1] + "…"

def display_path(value):
    path = Path(value).resolve()
    try:
        return str(path.relative_to(ROOT))
    except ValueError:
        return str(path)

def markdown_table(rows, columns):
    def safe(value):
        return one_line(value, 120).replace("|", "/")
    return "\n".join([
        "| " + " | ".join(label for _, label in columns) + " |",
        "|" + "|".join("---" for _ in columns) + "|",
        *("| " + " | ".join(safe(row.get(key, "")) for key, _ in columns) + " |"
          for row in rows),
    ])

preferred_run = (
    "20260827T135252029492Z_SYNX03_STORE_390_date_of_initial_diagnosis_policy_bundle"
)
run_id = os.environ.get("ACR_AUDIT_RUN_ID")
if run_id is None and ledger.selected_analysis(preferred_run):
    run_id = preferred_run
if run_id is None:
    selections = ledger.graph.find_nodes(node_type="AnalysisSelection")
    assert selections, "The ledger has no explicitly selected analysis"
    run_id = str((selections[-1].get("metadata") or {}).get("run_id"))
analysis_id = ledger.selected_analysis(run_id)
assert analysis_id, "Choose/select one reconstruction before human review"
run_dir = LEDGER_PATH.parent / run_id
artifact = ledger.load_analysis_artifact(run_id, analysis_id)
view = human_review_view(ledger, run_id, analysis_id, run_dir=run_dir)
display({
    "run_id": run_id,
    "analysis_id": analysis_id,
    "case": artifact.get("patient_id") or run_id.split("_", 2)[1],
    "task_arm": artifact.get("task_arm"),
    "review_model": artifact.get("review_model"),
    "reconstructor": artifact.get("reconstructor_identity"),
})


{'run_id': '20260827T135252029492Z_SYNX03_STORE_390_date_of_initial_diagnosis_policy_bundle',
 'analysis_id': 'analysis-27cf00959e3f0ce9ff2c',
 'case': 'SYNX03',
 'task_arm': 'policy_bundle',
 'review_model': 'openai/gpt-5.6-luna',
 'reconstructor': 'openrouter/openai/gpt-5.6-luna'}

## 2. Measure the abstraction ladder

These counts answer “how much does the reviewer have to read by default?” They do not
prove the abstraction is lossless. The fidelity check comes later: each Decision
Episode must preserve drill-down links and every deterministic cycle must be assigned
exactly once as decision-bearing, decision-support, or mechanical.


In [2]:
protocol_records = [
    json.loads(line) for line in (run_dir / "layer2_codex.jsonl").read_text().splitlines()
    if line.strip()
]
layer1_events = [
    json.loads(line) for line in (run_dir / "trace.jsonl").read_text().splitlines()
    if line.strip()
]
cycles = artifact["cycles"]
episodes = view["episodes"]
steps = view["review_chain"]["steps"]
ladder = [
    {"representation": "Codex protocol", "units": len(protocol_records),
     "default human use": "Harness/debug only"},
    {"representation": "Canonical Langtrace events", "units": len(layer1_events),
     "default human use": "Observable execution evidence"},
    {"representation": "Deterministic ReAct cycles", "units": len(cycles),
     "default human use": "State/action replay"},
    {"representation": "Decision Episodes", "units": len(episodes),
     "default human use": "Primary audit units"},
]
display(Markdown(markdown_table(ladder, [
    ("representation", "Representation"), ("units", "Units"),
    ("default human use", "Role"),
])))


| Representation | Units | Role |
|---|---|---|
| Codex protocol | 158 | Harness/debug only |
| Canonical Langtrace events | 22 | Observable execution evidence |
| Deterministic ReAct cycles | 21 | State/action replay |
| Decision Episodes | 8 | Primary audit units |

## 3. Read the observable raw trace

The protocol stream contains SDK lifecycle and model transport events. We count its
event types but deliberately do not print model reasoning payloads. The canonical
Layer-1 stream below is the useful raw audit record: tool name, compact action, and
server result shape.


In [3]:
protocol_types = Counter(str(row.get("type") or row.get("method") or "other")
                         for row in protocol_records)
display(Markdown("**Codex protocol event types (content redacted):** `" +
                 json.dumps(dict(protocol_types.most_common()), ensure_ascii=False) + "`"))

def event_action(event):
    payload = event.get("payload") or {}
    args = payload.get("args") or event.get("args") or {}
    tool = payload.get("tool") or event.get("tool") or ""
    for key in ("decision", "query", "note_id", "standing", "objective", "status"):
        if args.get(key) is not None:
            return f"{key}={args[key]}"
    return ""

raw_rows = []
for event in layer1_events:
    payload = event.get("payload") or {}
    result = payload.get("result") or event.get("result") or {}
    raw_rows.append({
        "seq": event.get("seq"),
        "kind": event.get("kind"),
        "tool": payload.get("tool") or event.get("tool") or "—",
        "action": event_action(event),
        "result": ", ".join(sorted(result)[:6]) if isinstance(result, dict) else "",
    })
display(Markdown(markdown_table(raw_rows, [
    ("seq", "Seq"), ("kind", "Kind"), ("tool", "Tool"),
    ("action", "Compact action"), ("result", "Result fields"),
])))


**Codex protocol event types (content redacted):** `{"item.agentMessage.delta": 76, "item.started": 33, "item.completed": 33, "thread.tokenUsage.updated": 13, "thread.started": 1, "turn.started": 1, "turn.completed": 1}`

| Seq | Kind | Tool | Compact action | Result fields |
|---|---|---|---|---|
| 1 | run_meta | — |  |  |
| 2 | tool_call | note_decision | decision=Perform the required unfiltered document inventory. | basis_sources, checked_fact_resolutions, citation_resolutions, context, decision_receipt, n_decisions |
| 3 | tool_call | list_documents | objective=Identify the complete set of patient documents and dates/types relevant to determining the first physician di… | context, documents, limit, objective, offset, page_complete |
| 4 | tool_call | note_decision | decision=Search the chart using the keyword batch diagnosis, cancer, malignancy, and carcinoma to locate candidate diag… | basis_sources, checked_fact_resolutions, citation_resolutions, context, decision_receipt, n_decisions |
| 5 | tool_call | search | query=diagnos | context, hits, n, objective |
| 6 | tool_call | search | query=cancer | context, hits, n, objective |
| 7 | tool_call | search | query=malignan | context, hits, n, objective |
| 8 | tool_call | search | query=carcinoma | context, hits, n, objective |
| 9 | tool_call | list_documents | objective=Complete the chart inventory for documents dated after the first inventory page, especially records surroundi… | context, documents, limit, objective, offset, page_complete |
| 10 | tool_call | read | note_id=Surgical-Pathology-Report_2022-03-09 | context, date, doc_type, note_id, objective, offset |
| 11 | tool_call | read | note_id=Surgical-Pathology-Document_2022-02-14 | context, date, doc_type, note_id, objective, offset |
| 12 | tool_call | read | note_id=Onc-Med-MD-OP-Progress-Note_2022-03-11 | context, date, doc_type, note_id, objective, offset |
| 13 | tool_call | record_finding | note_id=Surgical-Pathology-Document_2022-02-14 | basis_sources, checked_fact_resolutions, citation_resolutions, context, decision_receipt, finding_ref |
| 14 | tool_call | record_finding | note_id=Surgical-Pathology-Report_2022-03-09 | basis_sources, checked_fact_resolutions, citation_resolutions, context, decision_receipt, finding_ref |
| 15 | tool_call | record_finding | note_id=Onc-Med-MD-OP-Progress-Note_2022-03-11 | basis_sources, checked_fact_resolutions, citation_resolutions, context, decision_receipt, finding_ref |
| 16 | tool_call | note_decision | decision=Use 2022-03-09 as the initial diagnosis date; treat the 2022-02-14 cytology as non-establishing and the 2022-0… | basis_sources, checked_fact_resolutions, citation_resolutions, context, decision_receipt, n_decisions |
| 17 | tool_call | record_evidence | note_id=Surgical-Pathology-Document_2022-02-14 | n_evidence, quote, recorded |
| 18 | tool_call | record_evidence | note_id=Surgical-Pathology-Report_2022-03-09 | n_evidence, quote, recorded |
| 19 | tool_call | record_evidence | note_id=Onc-Med-MD-OP-Progress-Note_2022-03-11 | n_evidence, quote, recorded |
| 20 | tool_call | note_decision | decision=Stop chart review and submit the structured positive answer with explicit non-imputation flags. | basis_sources, checked_fact_resolutions, citation_resolutions, context, decision_receipt, n_decisions |
| 21 | tool_call | submit_answer | status=FOUND | accepted, kind, n_evidence, why |
| 22 | answer_accepted | — |  |  |

## 4. Replay the fixed ReAct cycles

A cycle is not automatically a decision. Search calls executing one precommitted
keyword batch can be support; pagination can be mechanical; a `record_finding` that
commits one note's Standing is decision-bearing. Reconstruction may label this fixed
skeleton, but it may not add, remove, reorder, duplicate, or move a cycle.


In [4]:
annotations = artifact["cycle_annotations"]
if isinstance(annotations, list):
    annotations = {row["cycle_id"]: row for row in annotations}
cycle_rows = []
for index, cycle in enumerate(cycles, 1):
    annotation = annotations[cycle["cycle_id"]]
    observed = (cycle.get("state_after") or {}).get("observed_state") or {}
    tools = [str(action.get("tool") or "") for action in cycle.get("actions") or []]
    cycle_rows.append({
        "n": index,
        "cycle": cycle["cycle_id"].rsplit(":", 1)[-1],
        "role": annotation["role"],
        "function": annotation.get("decision_function") or "—",
        "tools": ", ".join(tools) or "—",
        "receipt": "yes" if cycle.get("has_decision_receipt") else "no",
        "state": (f"surfaced={len(observed.get('surfaced_notes') or [])}; "
                  f"read={len(observed.get('read_notes') or [])}; "
                  f"findings={len((cycle.get('state_after') or {}).get('declared_state', {}).get('findings') or [])}"),
    })
display(Markdown(markdown_table(cycle_rows, [
    ("n", "#"), ("cycle", "Cycle"), ("role", "Role"),
    ("function", "Decision function"), ("tools", "Actions"),
    ("receipt", "Sealed receipt"), ("state", "State after"),
])))


| # | Cycle | Role | Decision function | Actions | Sealed receipt | State after |
|---|---|---|---|---|---|---|
| 1 | 2 | DECISION_BEARING | where_to_look | note_decision | yes | surfaced=0; read=0; findings=0 |
| 2 | 3 | DECISION_SUPPORT | — | list_documents | no | surfaced=200; read=0; findings=0 |
| 3 | 4 | DECISION_BEARING | where_to_look | note_decision | yes | surfaced=200; read=0; findings=0 |
| 4 | 5 | DECISION_SUPPORT | — | search | no | surfaced=201; read=0; findings=0 |
| 5 | 6 | DECISION_SUPPORT | — | search | no | surfaced=201; read=0; findings=0 |
| 6 | 7 | DECISION_SUPPORT | — | search | no | surfaced=202; read=0; findings=0 |
| 7 | 8 | DECISION_SUPPORT | — | search | no | surfaced=203; read=0; findings=0 |
| 8 | 9 | MECHANICAL | — | list_documents | no | surfaced=312; read=0; findings=0 |
| 9 | 10 | DECISION_BEARING | where_to_look | read | no | surfaced=312; read=1; findings=0 |
| 10 | 11 | DECISION_SUPPORT | — | read | no | surfaced=312; read=2; findings=0 |
| 11 | 12 | DECISION_SUPPORT | — | read | no | surfaced=312; read=3; findings=0 |
| 12 | 13 | DECISION_BEARING | standing | record_finding | yes | surfaced=312; read=3; findings=1 |
| 13 | 14 | DECISION_BEARING | standing | record_finding | yes | surfaced=312; read=3; findings=2 |
| 14 | 15 | DECISION_BEARING | standing | record_finding | yes | surfaced=312; read=3; findings=3 |
| 15 | 16 | DECISION_BEARING | which_wins | note_decision | yes | surfaced=312; read=3; findings=3 |
| 16 | 17 | DECISION_SUPPORT | — | record_evidence | no | surfaced=312; read=3; findings=3 |
| 17 | 18 | DECISION_SUPPORT | — | record_evidence | no | surfaced=312; read=3; findings=3 |
| 18 | 19 | DECISION_SUPPORT | — | record_evidence | no | surfaced=312; read=3; findings=3 |
| 19 | 20 | DECISION_BEARING | enough | note_decision | yes | surfaced=312; read=3; findings=3 |
| 20 | 21 | DECISION_SUPPORT | — | submit_answer | no | surfaced=312; read=3; findings=3 |
| 21 | 22 | MECHANICAL | — | — | no | surfaced=312; read=3; findings=3 |

## 5. Follow the human Decision Chain

Read this as if a colleague were explaining the review aloud. At each step ask:

1. Was this the right question at this point?
2. Were the meaningful alternatives represented?
3. Does the evidence/rule actually support the choice?
4. What judgment remained for the model?
5. If this step is wrong, which later steps inherit the problem?

A resolved policy reference proves only that the reference existed. It does not prove
semantic entailment or clinical correctness.


In [5]:
narrative = ["## The run, one consequential choice at a time"]
for index, step in enumerate(steps, 1):
    grounding = step.get("grounding_assessment") or {}
    flags = ", ".join(item["code"] for item in step.get("review_attention") or []) or "none"
    policy_refs = [
        str(item["rule_id"]) for item in step.get("guidelines") or []
        if item.get("rule_id")
    ]
    narrative.extend([
        f"### {index}. {step['phase_label']} — `{step['decision_function']}/{step['decision_subject']}`",
        f"- **Question before acting:** {step.get('question')}",
        f"- **Choice:** {step.get('decision')}",
        f"- **Stated reason:** {step.get('reason')}",
        f"- **Basis:** {', '.join(step.get('basis_sources') or []) or 'not recorded'}",
        f"- **Applied/offered clause refs shown here:** {', '.join(sorted(set(policy_refs))) or 'none'}",
        f"- **Reference status:** {grounding.get('reference_resolution_status')}",
        f"- **Remaining judgment:** {grounding.get('judgment_mode')}",
        f"- **Review attention:** {flags}",
        f"- **Resulting state:** {step.get('state_result')}",
        "",
    ])
conclusion = view["review_chain"]["conclusion"]
narrative.extend([
    "## Accepted conclusion",
    f"`{json.dumps(conclusion.get('value'), ensure_ascii=False)}`",
    f"\nSubmission explanation: {conclusion.get('reasoning')}",
])
display(Markdown("\n".join(narrative)))


## The run, one consequential choice at a time
### 1. Find the evidence — `where_to_look/retrieval_inventory`
- **Question before acting:** The diagnosis date and relevant source documents are unknown; the chart must be inventoried before deciding whether qualifying evidence exists.
- **Choice:** Perform the required unfiltered document inventory.
- **Stated reason:** The chart-review operating instruction requires list_documents without filters before asserting that the chart does not establish something.
- **Basis:** operational_instruction
- **Applied/offered clause refs shown here:** none
- **Reference status:** ALL_REFERENCES_RESOLVED
- **Remaining judgment:** OPERATIONAL_INSTRUCTION_APPLICATION
- **Review attention:** INVENTORY_PAGE_PARTIAL
- **Resulting state:** {'surfaced_notes': 200, 'read_notes': 0, 'findings': 0, 'uncertainties': 0}

### 2. Find the evidence — `where_to_look/retrieval_query_batch`
- **Question before acting:** The inventory shows oncology and physician progress notes beginning 2012-2013, plus surgical pathology in 2022; the first diagnosis may be documented in an earlier physician note or pathology record.
- **Choice:** Search the chart using the keyword batch diagnosis, cancer, malignancy, and carcinoma to locate candidate diagnosis statements and retrospective dates.
- **Stated reason:** These terms directly target physician diagnostic impressions and pathology/oncology references; the inventory decision identifies the relevant document families and time span.
- **Basis:** chart, task_contract
- **Applied/offered clause refs shown here:** decision_rule.1, evidence_rule.counts_as_evidence.1, evidence_rule.counts_as_evidence.2
- **Reference status:** ALL_REFERENCES_RESOLVED
- **Remaining judgment:** POLICY_GUIDED_OPERATIONAL_JUDGMENT
- **Review attention:** PROVISIONAL_INFERENCE_USED, RULE_APPLICATION_REVIEW
- **Resulting state:** {'surfaced_notes': 203, 'read_notes': 0, 'findings': 0, 'uncertainties': 0}

### 3. Find the evidence — `where_to_look/retrieval_document_set`
- **Question before acting:** Which candidate document set should be opened after the search results?
- **Choice:** Open selected document set: Surgical-Pathology-Report_2022-03-09, Surgical-Pathology-Document_2022-02-14, Onc-Med-MD-OP-Progress-Note_2022-03-11
- **Stated reason:** The first read after the search was a substantive selection of the candidate document set; cycles 11–12 opened the remaining members of that committed set.
- **Basis:** not recorded
- **Applied/offered clause refs shown here:** none
- **Reference status:** NO_REFERENCES_CLAIMED
- **Remaining judgment:** POLICY_GUIDED_OPERATIONAL_JUDGMENT
- **Review attention:** none
- **Resulting state:** {'surfaced_notes': 312, 'read_notes': 3, 'findings': 0, 'uncertainties': 0}

### 4. Judge the evidence — `standing/evidence_item`
- **Question before acting:** This pathology document is dated 2022-02-14 and reports atypical cells suspicious for squamous cell carcinoma with biopsy correlation recommended; its standing for the diagnosis date is unresolved.
- **Choice:** merely_mentions
- **Stated reason:** The document uses only an ambiguous suspicious term, so it does not itself establish the diagnosis under the evidence rule; a later positive biopsy and separate physician impression must be compared.
- **Basis:** chart, task_contract
- **Applied/offered clause refs shown here:** conflict_rule.2, evidence_rule.does_not_count.1
- **Reference status:** ALL_REFERENCES_RESOLVED
- **Remaining judgment:** POLICY_APPLICATION_JUDGMENT
- **Review attention:** none
- **Resulting state:** {'surfaced_notes': 312, 'read_notes': 3, 'findings': 1, 'uncertainties': 1}

### 5. Judge the evidence — `standing/evidence_item`
- **Question before acting:** This pathology report is dated 2022-03-09 and states that a right lower lobe biopsy demonstrates squamous cell carcinoma; its standing must be judged independently.
- **Choice:** can_establish
- **Stated reason:** A pathology interpretation with a final diagnosis of squamous cell carcinoma is a qualifying witness and establishes the diagnosis on its service date.
- **Basis:** chart, task_contract
- **Applied/offered clause refs shown here:** evidence_rule.counts_as_evidence.1
- **Reference status:** ALL_REFERENCES_RESOLVED
- **Remaining judgment:** POLICY_APPLICATION_JUDGMENT
- **Review attention:** none
- **Resulting state:** {'surfaced_notes': 312, 'read_notes': 3, 'findings': 2, 'uncertainties': 1}

### 6. Judge the evidence — `standing/evidence_item`
- **Question before acting:** This physician oncology note is dated 2022-03-11 and states that the right lower lobe mass clinically represents malignancy; its standing must be judged independently.
- **Choice:** can_establish
- **Stated reason:** A physician's clinical impression of malignancy qualifies as a clinical diagnosis without tissue, although this date is later than the confirmed biopsy date.
- **Basis:** chart, task_contract
- **Applied/offered clause refs shown here:** decision_rule.1, evidence_rule.counts_as_evidence.2
- **Reference status:** ALL_REFERENCES_RESOLVED
- **Remaining judgment:** POLICY_APPLICATION_JUDGMENT
- **Review attention:** none
- **Resulting state:** {'surfaced_notes': 312, 'read_notes': 3, 'findings': 3, 'uncertainties': 1}

### 7. Resolve the evidence — `which_wins/evidence_relationship`
- **Question before acting:** Three candidate findings are available: ambiguous cytology on 2022-02-14, confirmed biopsy on 2022-03-09, and physician clinical impression on 2022-03-11. The question is which date satisfies the contract's first-diagnosis rules.
- **Choice:** Use 2022-03-09 as the initial diagnosis date; treat the 2022-02-14 cytology as non-establishing and the 2022-03-11 physician impression as later corroboration.
- **Stated reason:** The cytology is ambiguous and lacks a same-date independent physician impression, so conflict_rule.2 directs use of the confirmatory biopsy date. The biopsy interpretation qualifies and predates the physician note; no earlier qualifying witness was found in the searched chart.
- **Basis:** chart, task_contract
- **Applied/offered clause refs shown here:** conflict_rule.2, decision_rule.1, evidence_rule.counts_as_evidence.1, evidence_rule.does_not_count.1
- **Reference status:** ALL_REFERENCES_RESOLVED
- **Remaining judgment:** POLICY_APPLICATION_JUDGMENT
- **Review attention:** none
- **Resulting state:** {'surfaced_notes': 312, 'read_notes': 3, 'findings': 3, 'uncertainties': 1}

### 8. Decide whether evidence is enough — `enough/case_sufficiency`
- **Question before acting:** The comparison resolved the diagnosis date as 2022-03-09, and qualifying and conflicting evidence spans have been recorded for the 2022-02-14 cytology, 2022-03-09 biopsy, and 2022-03-11 physician impression.
- **Choice:** Stop chart review and submit the structured positive answer with explicit non-imputation flags.
- **Stated reason:** A qualifying pathology witness with an exact date is recorded, all date-changing candidate sources were cited and resolved, and the date is fully explicit.
- **Basis:** chart, task_contract
- **Applied/offered clause refs shown here:** field_format.date_of_initial_diagnosis, proof_obligation.for_positive
- **Reference status:** ALL_REFERENCES_RESOLVED
- **Remaining judgment:** POLICY_APPLICATION_JUDGMENT
- **Review attention:** none
- **Resulting state:** {'surfaced_notes': 312, 'read_notes': 3, 'findings': 3, 'uncertainties': 1}

## Accepted conclusion
`{"date_of_initial_diagnosis": "20220309", "day_imputed": false, "month_imputed": false, "year_imputed": false}`

Submission explanation: The 2022-02-14 lung FNA was only suspicious and recommended biopsy, with no same-date independent physician cancer impression. The 2022-03-09 right lower lobe biopsy definitively diagnosed squamous cell carcinoma, and the 2022-03-11 physician note provided later clinical corroboration. Under the first-diagnosis and ambiguous-cytology conflict rules, the diagnosis date is 20220309.

## 6. Drill one flagged Decision back into evidence

We choose the first step with a review-attention flag. The compact chain tells us where
to look; the episode, runtime testimony, raw events, and field provenance tell us what
authority each statement has. This is the safeguard against a fluent reconstruction
hiding an execution error.


In [6]:
flagged_step = next((step for step in steps if step.get("review_attention")), steps[0])
episode_id = flagged_step["episode_ids"][0]
episode = next(row for row in episodes if row["episode_id"] == episode_id)
artifact_episode = next(
    row for row in artifact["episodes"] if row["episode_id"] == episode_id
)
reconstructed = episode["reconstruction"]
source_event_ids = set(artifact_episode.get("source_event_ids") or [])
source_events = [
    event for event in layer1_events if f"layer1:{event.get('seq')}" in source_event_ids
]
testimony = (episode.get("runtime_testimonies") or [{}])[0]
drill = {
    "question": flagged_step.get("question"),
    "choice": flagged_step.get("decision"),
    "runtime_testimony_ref": testimony.get("testimony_ref"),
    "runtime_because": testimony.get("because"),
    "reconstructed_rationale": reconstructed.get("decision_rationale"),
    "field_provenance": reconstructed.get("field_provenance"),
    "raw_event_ids": sorted(source_event_ids),
    "raw_tools": [((row.get("payload") or {}).get("tool") or row.get("tool"))
                  for row in source_events],
    "review_attention": flagged_step.get("review_attention"),
}
display(drill)
assert episode["bearing_cycle_id"]
assert episode["raw_langtrace_links"]


{'question': 'The diagnosis date and relevant source documents are unknown; the chart must be inventoried before deciding whether qualifying evidence exists.',
 'choice': 'Perform the required unfiltered document inventory.',
 'runtime_testimony_ref': 'decision:2',
 'runtime_because': 'The chart-review operating instruction requires list_documents without filters before asserting that the chart does not establish something.',
 'reconstructed_rationale': 'The chart-review operating instruction requires list_documents without filters before asserting that the chart does not establish something.',
 'field_provenance': {'candidate_set': 'SELF_REPORTED',
  'claimed_basis_summary': 'SELF_REPORTED',
  'counterfactual_supported_impact': 'MODEL_RECONSTRUCTED',
  'decision': 'SELF_REPORTED',
  'decision_rationale': 'SELF_REPORTED',
  'decision_subject': 'DETERMINISTIC_DERIVED_FROM_EXECUTION',
  'hypothesized_impact': 'MODEL_RECONSTRUCTED',
  'material_question': 'SELF_REPORTED',
  'model_interpr

## 7. Prove the abstraction still indexes the complete cycle skeleton

“Fewer units” is useful only if it does not silently drop a decision-bearing cycle.
The verifier requires every cycle exactly once across Decision Episodes and mechanical
cycles. Separately, every episode must link back to raw trace material.


In [7]:
episode_cycle_ids = [
    cycle_id for row in artifact["episodes"] for cycle_id in row["source_cycle_ids"]
]
mechanical_cycle_ids = artifact["mechanical_cycle_ids"]
all_cycle_ids = [row["cycle_id"] for row in cycles]
assert len(episode_cycle_ids + mechanical_cycle_ids) == len(all_cycle_ids)
assert set(episode_cycle_ids + mechanical_cycle_ids) == set(all_cycle_ids)
assert len(set(episode_cycle_ids + mechanical_cycle_ids)) == len(all_cycle_ids)
traceable = sum(bool(row["bearing_cycle_id"]) and bool(row["raw_langtrace_links"])
                for row in episodes)
assert traceable == len(episodes)
display({
    "cycles_accounted_for_exactly_once": f"{len(all_cycle_ids)}/{len(all_cycle_ids)}",
    "episodes_with_raw_drilldown": f"{traceable}/{len(episodes)}",
    "mechanical_cycles": len(mechanical_cycle_ids),
    "decision_or_support_cycles": len(episode_cycle_ids),
})


{'cycles_accounted_for_exactly_once': '21/21',
 'episodes_with_raw_drilldown': '8/8',
 'mechanical_cycles': 2,
 'decision_or_support_cycles': 19}

## 8. Compare task-only with policy-bundle behavior on the same case

If the historical experiment ledger is available, this section compares two selected
SYN0001 runs. It does not align steps merely by sequence number; it shows the semantic
function/subject, the chosen outcome, and whether the Decision node had a direct
Semantica `APPLIED_POLICY` binding.

The interesting result is not only that the final dates differ. The detailed run
retrieved and judged a same-day physician note that the task-only run did not use as
establishing evidence. That creates an actionable retrieval/standing/conflict audit
question.


In [8]:
comparison_path = ROOT / "runs/policy-experiment-20260827/experiment-ledger.json"
comparison_rows = []
if comparison_path.is_file():
    comparison_ledger = SemanticaLedger(comparison_path)
    comparison_runs = [
        "20260827T101823421486Z_SYN0001_STORE_390_date_of_initial_diagnosis_task_only",
        "20260827T105131502195Z_SYN0001_STORE_390_date_of_initial_diagnosis_policy_bundle",
    ]
    edge_rows = [edge.to_dict() if hasattr(edge, "to_dict") else dict(edge)
                 for edge in comparison_ledger.graph.edges]
    for candidate_run in comparison_runs:
        selected = comparison_ledger.selected_analysis(candidate_run)
        candidate_view = human_review_view(
            comparison_ledger,
            candidate_run,
            selected,
            run_dir=comparison_path.parent / candidate_run,
        )
        arm = candidate_view["task_presentation"]["arm_id"]
        for index, step in enumerate(candidate_view["review_chain"]["steps"], 1):
            decision_ids = {
                row["semantica_decision_id"] for row in step["detail_episodes"]
            }
            direct_bindings = sum(
                edge.get("type") == "APPLIED_POLICY"
                and edge.get("source_id") in decision_ids
                for edge in edge_rows
            )
            comparison_rows.append({
                "arm": arm,
                "step": index,
                "point": f"{step['decision_function']}/{step['decision_subject']}",
                "choice": step.get("decision"),
                "policy_bindings": direct_bindings,
            })
    display(Markdown(markdown_table(comparison_rows, [
        ("arm", "Arm"), ("step", "#"), ("point", "Decision point"),
        ("choice", "Choice"), ("policy_bindings", "Direct policy bindings"),
    ])))
else:
    display(Markdown(
        "Historical paired cohort not present. Notebook 1 can generate equivalent "
        "task-only and policy-bundle runs for comparison."
    ))


| Arm | # | Decision point | Choice | Direct policy bindings |
|---|---|---|---|---|
| task_only | 1 | where_to_look/retrieval_inventory | Inventory all patient documents before assessing whether the chart can establish the diagnosis date. | 0 |
| task_only | 2 | where_to_look/retrieval_query_batch | Run one precommitted diagnosis-focused keyword batch: diagnosis, diagnosed, cancer, tumour, tumor, neoplasm, pathology,… | 0 |
| task_only | 3 | where_to_look/retrieval_query_batch | Search specifically for adenocarcinoma and lung-cancer terminology to locate any earlier or retrospective physician dia… | 0 |
| task_only | 4 | where_to_look/retrieval_document_set | Read both pathology documents in full and assess each independently for the requested diagnosis date. | 0 |
| task_only | 5 | standing/evidence_item | Surgical-Pathology-Document_2023-04-12 is merely_mentions for date_of_initial_diagnosis (qualified_diagnostic_suspicion) | 0 |
| task_only | 6 | standing/evidence_item | Surgical-Pathology-Report_2023-04-27 is can_establish for date_of_initial_diagnosis (definitive_pathology_diagnosis) | 0 |
| task_only | 7 | which_wins/evidence_relationship | Use the 2023-04-27 definitive pathology report as the governing evidence and return 20230427 with no imputation. | 0 |
| policy_bundle | 1 | where_to_look/retrieval_inventory | Inventory all patient documents before making any negative or completeness claim, then target diagnostic evidence. | 2 |
| policy_bundle | 2 | where_to_look/retrieval_query_batch | Run a precommitted diagnostic keyword batch across the chart to locate physician cancer impressions, retrospective diag… | 7 |
| policy_bundle | 3 | where_to_look/retrieval_document_set | Read the two 2023-04-12 diagnostic documents and the 2023-04-27 positive pathology report as the selected candidate wit… | 4 |
| policy_bundle | 4 | standing/evidence_item | Surgical-Pathology-Document_2023-04-12 is merely_mentions for date_of_initial_diagnosis (ambiguous_cytology) | 2 |
| policy_bundle | 5 | standing/evidence_item | Onc-Med-MD-OP-Progress-Note_2023-04-12 is can_establish for date_of_initial_diagnosis (physician_clinical_diagnosis) | 3 |
| policy_bundle | 6 | standing/evidence_item | Surgical-Pathology-Report_2023-04-27 is can_establish for date_of_initial_diagnosis (positive_biopsy_pathology) | 2 |
| policy_bundle | 7 | which_wins/evidence_relationship | Use 2023-04-12 as the initial diagnosis date, because the physician's clinical impression establishes malignancy on tha… | 4 |

## 9. What the Decision layer gains and loses

| Gain | Corresponding risk | Safeguard in this notebook |
|---|---|---|
| One verdict per consequential choice | Reconstruction may choose the wrong boundary | Fixed cycles, sealed receipts, two-pass drift, explicit selection |
| Stable function/subject for cross-run comparison | A later taxonomy may reinterpret the run | Taxonomy is post-run and artifacts are append-only |
| Human-readable rationale and state transition | Fluent text may overstate what happened | Field provenance and exact source refs |
| Causal path and review routing | Temporal adjacency may be mistaken for causation | Only explicit evidenced causal assertions enter the audit chain |
| Much shorter default reading path | Incidental low-level errors may be hidden | 100% episode drill-down plus complete cycle accounting |

**Audit rule:** use the Decision Chain to decide *where to inspect*. Use the raw trace
and provenance to decide *what actually happened*. Clinical correctness still requires
a qualified reviewer.


In [9]:
closure = {
    "schema": "acr.postdoc_audit_walkthrough.v1",
    "run_id": run_id,
    "analysis_id": analysis_id,
    "counts": {
        "protocol_records": len(protocol_records),
        "langtrace_events": len(layer1_events),
        "react_cycles": len(cycles),
        "decision_episodes": len(episodes),
    },
    "all_cycles_accounted_for": True,
    "all_episodes_traceable": traceable == len(episodes),
    "priority_review_count": view["review_chain"]["priority_review_count"],
    "conclusion": view["review_chain"]["conclusion"],
}
output = ROOT / "runs/postdoc-notebook-output/02_audit_walkthrough.json"
output.parent.mkdir(parents=True, exist_ok=True)
output.write_text(json.dumps(closure, ensure_ascii=False, indent=2) + "\n")
display(Markdown(
    f"**Notebook 2 closed.** The human path has {len(episodes)} auditable choices; "
    f"all {len(episodes)} retain raw drill-down. Summary: `{display_path(output)}`."
))


**Notebook 2 closed.** The human path has 8 auditable choices; all 8 retain raw drill-down. Summary: `runs/postdoc-notebook-output/02_audit_walkthrough.json`.